# Colab A100 — SFT Readiness (Cursor Colab extension)

Full runbook: `docs/runbook_cursor_colab_ext_sft_readiness.md`

**Connect first:** kernel selector (top-right) → **Colab** → sign in as
`eric.wu@alphaaiengineering.com` → **Auto Connect** to the A100 (or *New Colab Server* → A100).
Then run cells top-to-bottom.

**Flow:** Steps 1–3 → ⏸ **PAUSE for approval before Step 4** (~9.85 GB pull) → Steps 5–6 → ⛔ **STOP at Step 7** (plan mode).

**Constraints:** frozen tokenizer is immutable; corpus (`data/`, `luts/`) is read-only; ask before the pull / any training / any HF push.

> Note: the extension runs these cells on the **remote A100**, but the filesystem is the remote VM (`/content`) — your local repo `.env` is *not* visible here (Step 3 handles the token).

## Step 1 — Verify runtime (no approval needed)
Confirm: an **A100** (40/80 GB), Python 3.1x, tens of GB free on `/content`.

In [1]:
!nvidia-smi
import sys; print("PY", sys.version)
!df -h /content

Thu Jul  9 19:36:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2 — Clone + install  ⚠ longish install
Use `[ml]` (torch/CUDA), **not** `[mlx]` (Apple-only).

In [6]:
!git clone https://github.com/ericrcwu001/SLM.git
!cd SLM && pip install -e '.[ml]'

Cloning into 'SLM'...
remote: Enumerating objects: 549, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 549 (delta 1), reused 6 (delta 1), pack-reused 531 (from 2)
Receiving objects: 100% (549/549), 223.18 MiB | 17.49 MiB/s, done.
Resolving deltas: 100% (136/136), done.
Updating files: 100% (325/325), done.
Obtaining file:///content/SLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.7 MB/s eta 0:00:00
  Building editable for slm-eval (pyproject.toml) ... done
  Created wheel for slm-eval: filename=slm_eval-0.1.0-0.editable-py3-none-any.whl size=6217 sha256=5b13a772d8a093e14c81b0771d317b00435acc148bb7e1609b52ec563159bb3a
  Stored

## Step 3 — HF auth (extension-ready; **replaces** `userdata.get`)
The extension has **no Secrets manager** and can't see your local repo `.env`. This loader resolves
`HF_TOKEN` from, in order: an existing env var → a `.env` **uploaded** to `/content/SLM/.env` or
`/content/.env` → a masked `getpass` paste. It never prints the token or bakes it into the notebook.

To skip the paste, upload the repo `.env` into `/content/SLM/` via Cursor's remote file explorer first.
⚠️ That file also holds Kaggle/freshluts secrets — upload an `HF_TOKEN`-only copy if you'd rather not
put those on the VM. A **read-only** token is enough for staging.

In [7]:
import os, getpass
from pathlib import Path

def _parse_env_file(path):
    vals = {}
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[len("export "):]
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        vals[k.strip()] = v.strip().strip('"').strip("'")
    return vals

def ensure_hf_token():
    if os.environ.get("HF_TOKEN"):
        return "already in os.environ"
    for p in ("/content/SLM/.env", "/content/.env", ".env"):
        if Path(p).exists():
            v = _parse_env_file(p).get("HF_TOKEN")
            if v:
                os.environ["HF_TOKEN"] = v
                return "loaded from " + p
    tok = getpass.getpass("Paste HF_TOKEN (input hidden; copy from repo .env): ").strip()
    if tok:
        os.environ["HF_TOKEN"] = tok
        return "loaded from getpass prompt"
    return None

src = ensure_hf_token()
print("HF_TOKEN source:", src)
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
assert os.environ.get("HF_TOKEN"), "HF_TOKEN not set - upload repo .env to the runtime or paste it." 

HF_TOKEN source: loaded from getpass prompt
HF_TOKEN set: True


## Step 4 — Stage the corpus  ⏸ DO NOT RUN until approved (~9.85 GB pull)
Keep Cursor connected for the whole pull. Confirm: **all 5 shards sha256-verified** and `/content/slm`
contains `luts/`, `data/`, `tokenizer/final/`.

In [8]:
# STOP: ~9.85 GB PULL - run only after explicit approval.
import os
!slm_stage stage --durable-root hf://datasets/ericrcwu/LUT_SLM --local-root /content/slm
os.environ["SLM_ARTIFACT_ROOT"] = "/content/slm"
print("SLM_ARTIFACT_ROOT =", os.environ["SLM_ARTIFACT_ROOT"])
!ls -la /content/slm

stage_manifest.json: 100% 1.29k/1.29k [00:00<00:00, 2.11MB/s]
corpus-0000.tar: 100% 2.16G/2.16G [00:42<00:00, 50.4MB/s]
[stage] corpus-0000.tar: verified + extracted 9133 files
corpus-0001.tar: 100% 2.15G/2.15G [00:41<00:00, 51.7MB/s]
[stage] corpus-0001.tar: verified + extracted 3316 files
corpus-0002.tar: 100% 2.16G/2.16G [01:10<00:00, 30.7MB/s]
[stage] corpus-0002.tar: verified + extracted 8065 files
corpus-0003.tar: 100% 2.15G/2.15G [00:52<00:00, 40.6MB/s]
[stage] corpus-0003.tar: verified + extracted 4096 files
corpus-0004.tar: 100% 1.95G/1.95G [01:01<00:00, 31.6MB/s]
[stage] corpus-0004.tar: verified + extracted 2253 files
[stage] staged to /content/slm  (set SLM_ARTIFACT_ROOT=/content/slm)
{
  "attempted": 5,
  "bytes": 10577479680,
  "command": "stage",
  "durable_root": "hf://datasets/ericrcwu/LUT_SLM",
  "note": "staged to /content/slm  (set SLM_ARTIFACT_ROOT=/content/slm)",
  "shards": [],
  "skipped": 0,
  "status": "ok",
  "transferred": 5,
  "verified": 5
}
SLM_ARTIFACT_R

## Step 5 — VERIFY THE FROZEN TOKENIZER (hard gate)
**If either assert fails → STOP and report. Do not proceed.**

In [9]:
import json, pathlib
d = pathlib.Path("/content/slm/tokenizer/final")
need = {"model.pt", "encoder.pt", "decoder.pt", "codebook.npy", "manifest.json"}
have = {p.name for p in d.iterdir()}
assert need <= have, f"MISSING weights: {sorted(need - have)}"

m = json.load(open(d / "manifest.json"))
EXPECT_VER = "vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f"
EXPECT_SHA = "bcdf369dd7cd9a99d71f240b0dac67d404f52130dc8c35d14d6a04514349d118"
assert m.get("tokenizer_version")  == EXPECT_VER, ("VERSION MISMATCH", m.get("tokenizer_version"))
assert m.get("vq_codebook_sha256") == EXPECT_SHA, ("SHA MISMATCH",     m.get("vq_codebook_sha256"))
print("FROZEN TOKENIZER OK:", m["tokenizer_version"])

FROZEN TOKENIZER OK: vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f


### Step 5b (optional) — copy staged tokenizer into the repo
Only if you prefer copying over resolving via `SLM_ARTIFACT_ROOT`. Ensures SFT/tokenize code loads the
**staged frozen** weights, never the weightless cloned stub.

In [ ]:
import shutil, pathlib
src = pathlib.Path("/content/slm/tokenizer/final")
dst = pathlib.Path("/content/SLM/tokenizer/final"); dst.mkdir(parents=True, exist_ok=True)
for p in src.iterdir(): shutil.copy2(p, dst / p.name)
print("copied:", sorted(x.name for x in dst.iterdir()))

## Step 6 — Read docs, scope tokenizer→SFT prerequisites (read-only)

In [10]:
for f in ["docs/training_plan_colab.md", "docs/master_plan.md", "docs/model_architecture.md"]:
    print("\n" + "="*80 + f"\n{f}\n" + "="*80)
    print(open(f"/content/SLM/{f}").read())


docs/training_plan_colab.md
# Training Plan Using Google Colab

## Objective

Train a small image-conditioned prompt-to-LUT model in Google Colab using the
full prompt-to-LUT architecture with caveats:

- full prompt-to-LUT architecture;
- active instruction SFT set around 10k-15k examples, not 50k for v1;
- generative LUT-token warmup before instruction SFT;
- CLI-first demo;
- workbench later;
- broad source collection but usage-aware, diversity-culled active training;
- QLoRA SFT first;
- RS/DPO before GRPO;
- GRPO only after simpler tuned stages pass behavior gates and plateau.

## Runtime Assumptions

Preferred Colab runtime:

```text
A100 or L4 GPU
```

T4 is acceptable for:

- dependency checks;
- tokenizer smoke tests;
- data validation;
- 50-example overfit run;
- tiny SFT smoke tests.

T4 is not the assumed runtime for full 10k-15k SFT.

## Notebook Structure

Recommended notebooks:

```text
notebooks/
  00_environment_check.ipynb
  01_lut_tokenizer_training.ipynb
  02_datas

## Step 7 — STOP for plan approval  ⛔
No training, no wiring stubs, no edits to `data/`/`luts/`. I'll produce the **readiness report** +
**first-SFT-run plan** (levers: cap `max_pixels`, raise batch size, keep `epochs=2`) in **plan mode**
for your approval before implementing anything.